# Direct-path neural analysis: alternation vs. switch

Same data source as `change_of_mind.ipynb` (`improved_node_summary_df` + spikes), but
here we look at **direct** (`port_to_port_path_type == 'direct'`) port-to-port paths
instead of long indirect ones. Direct paths split into two kinds:

- **Alternation**: between the two ports of the same patch ([1,2], [3,4], [5,6], [7,8])
- **Switch**: between ports belonging to two different patches

For each (from-port, to-port) pair we aggregate every direct traversal of that pair,
compute each good unit's firing rate at every node along the path, and average across
traversals (mean +/- SEM). Two normalizations are used:

1. **Within a pair**: traversals of the same pair are resampled (linear interpolation
   over fractional path position) onto that pair's most common ("canonical") node
   sequence, so the x-axis can show the literal node names. In this dataset every
   traversal of a given pair already has the identical node sequence (verified below),
   so this step is a no-op here -- but it keeps the pipeline robust if that's not true
   in other sessions.
2. **Across pairs** (for the combined Alternation / Switch overview plots): pairs have
   genuinely different lengths (e.g. alternation paths = 8 nodes, switch paths = 10-11
   nodes), so those are resampled onto a shared *normalized path position* axis
   (0 = source port, 1 = destination port) instead of node identity.

In [54]:
import scipy.io
import pandas as pd
import h5py
import numpy as np
import pickle
import ast
import re
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import glob

In [55]:
TARGET_COLORS = {
    'Target1': (84/255,  64/255,  204/255),
    'Target2': (128/255, 140/255, 255/255),
    'Target3': (204/255, 115/255, 51/255),
    'Target4': (255/255, 191/255, 140/255),
    'Target5': (89/255,  153/255, 64/255),
    'Target6': (140/255, 204/255, 166/255),
    'Target7': (178/255, 102/255, 140/255),
    'Target8': (217/255, 153/255, 230/255),
}

# For switch overview plots: cycle through these linestyles per unique from_label
SWITCH_FROM_LINESTYLES = ['-', '--', ':', '-.']

## Session setup

In [56]:
# Update the session variable to match the date in your file names
session = '10/09/2025'

YY = session[-2:]
YYYY = session[-4:]
MM = session[3:5]
DD = session[:2]

DD_MM_YYYY = f"{DD}_{MM}_{YYYY}"
MM_DD_YYYY = f"{MM}_{DD}_{YYYY}"


In [57]:
spike_clusters = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_clusters.mat')
spike_times = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_times.mat')
cluster_info = pd.read_csv(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/cluster_info.tsv', sep='\t')

In [58]:
improved_node_summary_csv = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/node_summary_df.csv"
summary_df_pkl = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/summary_df.pkl"

with open(improved_node_summary_csv, 'rb') as f:
    improved_node_summary_df = pd.read_csv(f)

with open(summary_df_pkl, 'rb') as f:
    summary_df = pickle.load(f)

improved_node_summary_df.shape

(5727, 17)

## Spike data

In [59]:
good_units = cluster_info[cluster_info['group'] == 'good'].copy().reset_index(drop=True)
mua_units  = cluster_info[cluster_info['group'] == 'mua'].copy().reset_index(drop=True)
# noise units are dropped entirely

print(f"Total clusters : {len(cluster_info)}")
print(f"Good units     : {len(good_units)}")
print(f"MUA units      : {len(mua_units)}")
print(f"Noise dropped  : {(cluster_info['group'] == 'noise').sum()}")

Total clusters : 1005
Good units     : 34
MUA units      : 137
Noise dropped  : 793


In [60]:
FPS = 40       # video frames per second
FS = 30_000    # Neuropixels sampling rate (Hz)

def _unwrap_mat(d):
    keys = [k for k in d if not k.startswith('__')]
    arr = d[keys[0]]
    return arr.flatten()

spike_times_all    = _unwrap_mat(spike_times)
spike_clusters_all = _unwrap_mat(spike_clusters)

# Maze trials are 31-766 (1-indexed) -> indices 30:767 (0-indexed, 737 trials)
MAZE_START = 30
MAZE_END   = 767

maze_spike_times    = spike_times_all[MAZE_START:MAZE_END]
maze_spike_clusters = spike_clusters_all[MAZE_START:MAZE_END]

assert len(maze_spike_times) == 737, f"Expected 737 maze trials, got {len(maze_spike_times)}"
print(f"Maze trials extracted: {len(maze_spike_times)} (trials 31-766)")

Maze trials extracted: 737 (trials 31-766)


In [61]:
# summary_df has MultiIndex columns (from the tracking pipeline); flatten the
# behavior-only columns we need to build a trial -> absolute-time lookup.
sdf = summary_df.copy()
sdf.columns = ['_'.join([c for c in col if c]) if isinstance(col, tuple) else col for col in sdf.columns]

trial_meta = sdf.groupby('trial_idx').agg(
    start_frame=('frame_idx_global', 'min'),
    trial_length=('trial_length', 'first'),
)
trial_meta['start_abs'] = trial_meta['start_frame'] / FPS
trial_meta['end_abs'] = trial_meta['start_abs'] + trial_meta['trial_length']

def trials_overlapping(t_lo, t_hi):
    """All pipeline 'trial' indices whose time range overlaps [t_lo, t_hi]."""
    m = (trial_meta['end_abs'] >= t_lo) & (trial_meta['start_abs'] <= t_hi)
    return trial_meta.index[m].tolist()

def gather_relative_spikes(trials_in_path, cluster_id, t0_abs):
    """Spikes for one cluster across the given trials, as seconds relative to t0_abs,
    by stitching each trial's spike times (stored relative to that trial's own start, in
    samples) onto the global video clock via trial_meta['start_abs']."""
    rel_chunks = []
    for t in trials_in_path:
        spike_t = np.asarray(maze_spike_times[t]).flatten() / FS
        spike_c = np.asarray(maze_spike_clusters[t]).flatten()
        abs_t = trial_meta.loc[t, 'start_abs'] + spike_t
        rel_chunks.append(abs_t[spike_c == cluster_id] - t0_abs)
    return np.concatenate(rel_chunks) if rel_chunks else np.array([])

## Step 1: Extract direct-path instances

One instance per contiguous run of a `path_pair_label` whose `port_to_port_path_type`
is `'direct'`. Each instance carries its full node sequence plus per-node start/end
times. `('TargetN', 'TargetM')` pairs are classified `alternation` if N and M are in
the same port-pair group ([1,2],[3,4],[5,6],[7,8]) and `switch` otherwise.

`start_frame`/`end_frame` in `improved_node_summary_df` are `frame_idx_global` values
(verified against `summary_df`), so they're trustworthy, trial-time-aligned frame
indices -- per-node timing here always derives from them directly:
`start_abs = start_frame / FPS`, `end_abs = (end_frame + 1) / FPS`. The stored
`duration_frames` column is **not** used: per `FRAME_OFFSET_BUG_ANALYSIS.md` /
`FRAME_OFFSET_FIX.md`, the upstream traversal-reconstruction step can steal frames
between adjacent runs when inferring a missing node, which desyncs `duration_frames`
(list-length-based) from the true `start_frame`/`end_frame` span for those rows --
`end_frame - start_frame + 1` is the reliable duration.

In [62]:
def parse_pair_label(label):
    try:
        return ast.literal_eval(label)
    except (ValueError, SyntaxError, TypeError):
        return (None, None)

def target_group(label):
    """Group index (0-3) for a 'TargetN' label: ports [1,2]->0, [3,4]->1, [5,6]->2, [7,8]->3."""
    n = int(re.search(r'\d+', str(label)).group())
    return (n - 1) // 2, n

df_sorted = improved_node_summary_df.sort_values('node_visit_idx').reset_index(drop=True)

# One row per contiguous run of identical path_pair_label (the row marking the start
# of each port-to-port path traversal).
path_segments = []
prev_label = None
for _, row in df_sorted.iterrows():
    label = row['path_pair_label']
    if pd.isna(label):
        prev_label = None
        continue
    if label != prev_label:
        path_segments.append(row)
    prev_label = label

print(f"Total port-to-port path segments in session: {len(path_segments)}")

direct_instances = []
for seg_row in path_segments:
    if seg_row['port_to_port_path_type'] != 'direct':
        continue

    start_nv = int(seg_row['node_visit_idx'])
    seq_len = int(seg_row['path_seq_length_unique'])
    end_nv = start_nv + seq_len

    seq_rows = improved_node_summary_df[
        (improved_node_summary_df['node_visit_idx'] >= start_nv) &
        (improved_node_summary_df['node_visit_idx'] <= end_nv)
    ].sort_values('node_visit_idx').reset_index(drop=True)

    if seq_rows.empty:
        continue

    from_label, to_label = parse_pair_label(seg_row['path_pair_label'])
    from_grp, _ = target_group(from_label)
    to_grp, _ = target_group(to_label)
    path_class = 'alternation' if from_grp == to_grp else 'switch'

    node_seq = seq_rows['node_name'].tolist()
    start_frames = seq_rows['start_frame'].to_numpy()
    end_frames = seq_rows['end_frame'].to_numpy()
    starts_abs = start_frames / FPS
    ends_abs = (end_frames + 1) / FPS
    durations_sec = ends_abs - starts_abs

    direct_instances.append({
        'pair_label': (from_label, to_label),
        'path_class': path_class,
        'trial_idx': int(seg_row['trial_idx']),
        'node_seq': node_seq,
        'n_nodes': len(node_seq),
        'starts_abs': starts_abs,
        'ends_abs': ends_abs,
        'durations_sec': durations_sec,
        'path_start_abs': starts_abs[0],
        'path_end_abs': ends_abs[-1],
        # Raw global frame boundaries of the whole path (first node's start_frame,
        # last node's end_frame) -- used by the edge-inclusive analysis below to slice
        # the matching rows out of traversal_df, which has no direct/indirect labeling
        # of its own.
        'path_frame_start': int(start_frames[0]),
        'path_frame_end': int(end_frames[-1]),
    })

print(f"Direct path instances: {len(direct_instances)}")

pairs_to_instances = {}
for inst in direct_instances:
    pairs_to_instances.setdefault(inst['pair_label'], []).append(inst)

print(f"Unique direct (from, to) pairs: {len(pairs_to_instances)}")

Total port-to-port path segments in session: 521
Direct path instances: 426
Unique direct (from, to) pairs: 30


In [63]:
# Sanity check: how many pairs have traversals of inconsistent node-sequence length?
pair_overview_rows = []
for pair_label, instances in pairs_to_instances.items():
    lengths = sorted(set(inst['n_nodes'] for inst in instances))
    pair_overview_rows.append({
        'pair_label': pair_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'unique_lengths': lengths,
    })

pair_overview_df = pd.DataFrame(pair_overview_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
n_inconsistent = (pair_overview_df['unique_lengths'].apply(len) > 1).sum()
print(f"Pairs with more than one observed node-sequence length: {n_inconsistent} (resampling handles these if present)")
display(pair_overview_df)

Pairs with more than one observed node-sequence length: 0 (resampling handles these if present)


,pair_label,path_class,n_instances,unique_lengths
0,"(Target4, Target3)",alternation,56,[9]
1,"(Target7, Target8)",alternation,53,[9]
2,"(Target8, Target7)",alternation,52,[9]
3,"(Target2, Target1)",alternation,47,[9]
4,"(Target3, Target4)",alternation,46,[9]
5,"(Target5, Target6)",alternation,38,[9]
6,"(Target6, Target5)",alternation,36,[9]
7,"(Target1, Target2)",alternation,30,[9]
8,"(Target2, Target8)",switch,12,[11]
9,"(Target5, Target4)",switch,7,[11]


## Step 2: Per-node firing rate + length normalization helpers

`compute_node_rates` gives one firing-rate value (Hz) per node visited along a single
path instance, for one unit. `resample_to_grid` linearly interpolates any such sequence
(or its mean/SEM) from its own length onto a target number of points, indexed by
fractional position along the path (0 = source port, 1 = destination port). This is
the single normalization primitive used both for aligning same-pair instances onto a
canonical node grid, and for aligning different-length pairs onto the shared overview
axis.

In [64]:
MIN_INSTANCES = 2  # need at least 2 traversals to compute a meaningful mean +/- SEM

def compute_node_rates(instance, cluster_id):
    """Firing rate (Hz) of one unit during each node-visit of one path instance."""
    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])
    rel_starts = instance['starts_abs'] - instance['path_start_abs']
    rel_ends = instance['ends_abs'] - instance['path_start_abs']

    rates = np.empty(instance['n_nodes'])
    for i in range(instance['n_nodes']):
        count = np.sum((rel_spikes >= rel_starts[i]) & (rel_spikes < rel_ends[i]))
        rates[i] = count / instance['durations_sec'][i]
    return rates

def resample_to_grid(values, n_grid):
    """Linearly interpolate `values` (length L) onto `n_grid` points over fractional
    position 0..1."""
    values = np.asarray(values, dtype=float)
    L = len(values)
    if L == 1:
        return np.full(n_grid, values[0])
    src_frac = np.linspace(0, 1, L)
    dst_frac = np.linspace(0, 1, n_grid)
    return np.interp(dst_frac, src_frac, values)

def canonical_sequence(instances):
    """Most common exact node sequence among a pair's traversals."""
    seqs = [tuple(inst['node_seq']) for inst in instances]
    most_common_seq, _ = Counter(seqs).most_common(1)[0]
    return list(most_common_seq)

## Step 3: Aggregate mean +/- SEM per (pair, good unit)

For every direct-path pair with at least `MIN_INSTANCES` traversals, and every good
unit, resample each traversal's per-node rate onto the pair's canonical node grid, then
take the mean and SEM across traversals. Cached in `pair_unit_stats` for reuse by both
the per-pair plots and the Alternation/Switch overview plots.

In [65]:
pair_unit_stats = {}  # (pair_label, cluster_id) -> dict(mean, sem, n, canonical_seq, path_class)
skipped_pairs = []

good_cluster_ids = good_units['cluster_id'].astype(int).tolist()

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        skipped_pairs.append((pair_label, len(instances)))
        continue

    canonical_seq = canonical_sequence(instances)
    L = len(canonical_seq)
    path_class = instances[0]['path_class']

    for cid in good_cluster_ids:
        resampled = np.vstack([
            resample_to_grid(compute_node_rates(inst, cid), L)
            for inst in instances
        ])
        mean = resampled.mean(axis=0)
        sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
        pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'canonical_seq': canonical_seq,
            'path_class': path_class,
        }

print(f"Computed stats for {len(pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")
print(f"Pairs skipped (fewer than {MIN_INSTANCES} traversals): {len(skipped_pairs)}")
for pl, n in skipped_pairs:
    print(f"  {pl}: n={n}")

Computed stats for 22 pairs x 34 good units
Pairs skipped (fewer than 2 traversals): 8
  ('Target8', 'Target6'): n=1
  ('Target8', 'Target1'): n=1
  ('Target1', 'Target5'): n=1
  ('Target4', 'Target6'): n=1
  ('Target4', 'Target1'): n=1
  ('Target2', 'Target7'): n=1
  ('Target2', 'Target5'): n=1
  ('Target4', 'Target5'): n=1


## Step 4: Per-pair plots

One figure per (pair, good unit): mean firing rate (+/- SEM shaded band) at each node
along the canonical path, x-axis labeled with the actual node names. Saved under
`resources/outputs/{session}/direct_path_neural/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [66]:
output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural")
output_root.mkdir(parents=True, exist_ok=True)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = output_root / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        stats = pair_unit_stats[(pair_label, cid)]
        canonical_seq = stats['canonical_seq']
        L = len(canonical_seq)
        mean, sem, n = stats['mean'], stats['sem'], stats['n']
        c = TARGET_COLORS.get(to_label, 'steelblue')

        fig, ax = plt.subplots(figsize=(max(6, L * 0.45), 4))
        x = np.arange(L)
        ax.plot(x, mean, color=c, linewidth=1.8)
        ax.fill_between(x, mean - sem, mean + sem, color=c, alpha=0.3, label='SEM')
        ax.set_xticks(x)
        ax.set_xticklabels(canonical_seq, rotation=90, fontsize=7)
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_xlabel('Path node (canonical sequence)')
        ax.set_title(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label} | n={n} traversals",
            fontsize=9,
        )
        ax.legend(fontsize=7, loc='upper right')
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair plots under {output_root}")

Saved 748 per-pair plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural


## Step 5: Alternation / Switch overview plots

One figure per good unit per path class, overlaying every pair of that class as its own
line (+/- SEM band), on a shared **normalized path-position** axis (0 = source port,
1 = destination port) since pairs of the same class can still have different lengths
(e.g. alternation paths in this session are all 8 nodes, but switch paths are 10-11 --
the normalized axis is what makes overlaying them meaningful). Saved under
`resources/outputs/{session}/direct_path_neural/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [67]:
OVERVIEW_GRID = 100
frac_grid = np.linspace(0, 1, OVERVIEW_GRID)

for path_class in ['alternation', 'switch']:
    class_dir = output_root / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For alternation, use a node-indexed x-axis if all paths share the same canonical length
    use_node_indexed = False
    node_x_vals = node_x_tick_positions = node_x_tick_labels = None
    if path_class == 'alternation' and relevant_pairs:
        lengths = set(len(pair_unit_stats[(pl, good_cluster_ids[0])]['canonical_seq']) for pl in relevant_pairs)
        if len(lengths) == 1:
            L_shared = lengths.pop()
            mid = (L_shared - 1) // 2
            node_x_vals = np.arange(L_shared)
            node_x_tick_positions = [0, mid, L_shared - 1]
            node_x_tick_labels = ['Source\nPort', 'Midpoint', 'Target\nPort']
            use_node_indexed = True

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, ax = plt.subplots(figsize=(8, 5))

        for pair_label in relevant_pairs:
            stats = pair_unit_stats[(pair_label, cid)]
            from_label, to_label = pair_label
            color = TARGET_COLORS.get(to_label, 'gray')
            ls = from_to_ls.get(from_label, '-')

            if use_node_indexed:
                mean_plot = stats['mean']
                sem_plot = stats['sem']
                x_plot = node_x_vals
            else:
                L = len(stats['canonical_seq'])
                mean_plot = resample_to_grid(stats['mean'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['mean']
                sem_plot = resample_to_grid(stats['sem'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['sem']
                x_plot = frac_grid

            ax.plot(x_plot, mean_plot, color=color, linewidth=1.5, linestyle=ls,
                     label=f"{from_label}->{to_label} (n={stats['n']})")
            ax.fill_between(x_plot, mean_plot - sem_plot, mean_plot + sem_plot,
                             color=color, alpha=0.15)

        if use_node_indexed:
            ax.set_xticks(node_x_tick_positions)
            ax.set_xticklabels(node_x_tick_labels)
            ax.set_xlabel('Path node position')
        else:
            ax.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(f"Unit {cid} | {path_class.capitalize()} direct paths", fontsize=10)
        ax.legend(fontsize=6, loc='upper right', ncol=2)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

Saved 34 'alternation' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/alternation_overview (8 pairs overlaid)
Saved 34 'switch' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/switch_overview (14 pairs overlaid)


## Save pair summary

In [68]:
summary_out_rows = []
for pair_label, instances in pairs_to_instances.items():
    from_label, to_label = pair_label
    summary_out_rows.append({
        'from_port': from_label,
        'to_port': to_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'n_nodes': instances[0]['n_nodes'],
        'used_in_aggregation': len(instances) >= MIN_INSTANCES,
        'canonical_sequence': ' -> '.join(canonical_sequence(instances)),
    })

direct_path_pair_summary_df = pd.DataFrame(summary_out_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
out_csv = output_root / 'direct_path_pair_summary.csv'
direct_path_pair_summary_df.to_csv(out_csv, index=False)
print(f"Saved pair summary to {out_csv}")
display(direct_path_pair_summary_df)

Saved pair summary to /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/direct_path_pair_summary.csv


,from_port,to_port,path_class,n_instances,n_nodes,used_in_aggregation,canonical_sequence
0,Target4,Target3,alternation,56,9,True,L510 -> L45 -> L32 -> L21 -> L10 -> L20 -> L31...
1,Target7,Target8,alternation,53,9,True,R59 -> R44 -> R32 -> R21 -> R10 -> R20 -> R31 ...
2,Target8,Target7,alternation,52,9,True,R55 -> R42 -> R31 -> R20 -> R10 -> R21 -> R32 ...
3,Target2,Target1,alternation,47,9,True,R526 -> R413 -> R36 -> R23 -> R11 -> R22 -> R3...
4,Target3,Target4,alternation,46,9,True,L56 -> L43 -> L31 -> L20 -> L10 -> L21 -> L32 ...
5,Target5,Target6,alternation,38,9,True,L521 -> L410 -> L35 -> L22 -> L11 -> L23 -> L3...
6,Target6,Target5,alternation,36,9,True,L525 -> L412 -> L36 -> L23 -> L11 -> L22 -> L3...
7,Target1,Target2,alternation,30,9,True,R522 -> R411 -> R35 -> R22 -> R11 -> R23 -> R3...
8,Target2,Target8,switch,12,11,True,R526 -> R413 -> R36 -> R23 -> R11 -> R0 -> R10...
9,Target5,Target4,switch,7,11,True,L521 -> L410 -> L35 -> L22 -> L11 -> L0 -> L10...


# Additional analysis: continuous-time version (node-agnostic)

The per-node analysis above only counts spikes while the mouse is parked *at* a node
-- it has no data during the "edge" transit gaps between consecutive node-visits, so
those gaps are silently skipped rather than contributing zero/low/high firing.

This version instead treats each path traversal as one continuous time window:
`start = first node's start_frame`, `end = last node's end_frame` (i.e. exactly
`path_start_abs` / `path_end_abs`, already computed above), and bins spikes across
the *entire* window at a fixed bin width, with no reference to node identity at all.

Unlike the per-node version, normalization here is *not* a no-op: even though every
traversal of a given pair visits the same fixed number of nodes, the total elapsed
time for the path (dwell times + transit gaps) still varies traversal to traversal.
So every instance's binned rate trace is resampled (same `resample_to_grid` helper as
above) onto a shared **normalized path-time** axis (0 = path start, 1 = path end)
before averaging across traversals of a pair, and again across pairs for the
Alternation/Switch overview.

This is purely additive -- it doesn't replace the per-node analysis, and writes to a
separate output folder (`direct_path_neural_continuous/`).

In [69]:
from scipy.ndimage import gaussian_filter1d

BIN_SIZE_CONTINUOUS = 0.05      # seconds, histogram bin width for the raw rate
SMOOTH_SIGMA_CONTINUOUS = 0.15  # seconds, stdev of the Gaussian smoothing kernel
CONTINUOUS_GRID = 100           # points on the shared normalized-path-time axis

def compute_continuous_rate(instance, cluster_id):
    """Binned + lightly smoothed firing rate (Hz) of one unit across the path's
    full time window (path_start_abs -> path_end_abs), independent of node identity."""
    duration = instance['path_end_abs'] - instance['path_start_abs']
    n_bins = max(1, round(duration / BIN_SIZE_CONTINUOUS))
    bin_edges = np.linspace(0, duration, n_bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]

    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])

    counts, _ = np.histogram(rel_spikes, bins=bin_edges)
    rate = counts.astype(float) / bin_width
    sigma_bins = SMOOTH_SIGMA_CONTINUOUS / bin_width
    return gaussian_filter1d(rate, sigma=sigma_bins, mode='constant')

## Aggregate mean +/- SEM per (pair, good unit), continuous-time version

Same `MIN_INSTANCES` threshold and pair grouping as the per-node analysis -- only the
within-pair resampling target changes (a fixed `CONTINUOUS_GRID` of normalized-time
points instead of the pair's canonical node count).

In [70]:
continuous_pair_unit_stats = {}  # (pair_label, cluster_id) -> dict(mean, sem, n, mean_duration, path_class)

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    path_class = instances[0]['path_class']
    mean_duration = np.mean([inst['path_end_abs'] - inst['path_start_abs'] for inst in instances])

    for cid in good_cluster_ids:
        resampled = np.vstack([
            resample_to_grid(compute_continuous_rate(inst, cid), CONTINUOUS_GRID)
            for inst in instances
        ])
        mean = resampled.mean(axis=0)
        sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
        continuous_pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'mean_duration': mean_duration,
            'path_class': path_class,
        }

print(f"Computed continuous-time stats for {len(continuous_pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")

Computed continuous-time stats for 22 pairs x 34 good units


## Per-pair plots (continuous-time)

X-axis is normalized path time (0 = path start, 1 = path end) since, unlike the
per-node version, raw seconds aren't comparable across traversals of the same pair.
The mean traversal duration is noted in the title for context. Saved under
`resources/outputs/{session}/direct_path_neural_continuous/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [ ]:
continuous_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_continuous")
continuous_output_root.mkdir(parents=True, exist_ok=True)

continuous_frac_grid = np.linspace(0, 1, CONTINUOUS_GRID)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = continuous_output_root / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        stats = continuous_pair_unit_stats[(pair_label, cid)]
        mean, sem, n = stats['mean'], stats['sem'], stats['n']
        c = TARGET_COLORS.get(to_label, 'darkorange')

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(continuous_frac_grid, mean, color=c, linewidth=1.8)
        ax.fill_between(continuous_frac_grid, mean - sem, mean + sem, color=c, alpha=0.1, label='SEM')
        ax.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label} | n={n} traversals, "
            f"mean duration={stats['mean_duration']:.2f}s",
            fontsize=9,
        )
        ax.legend(fontsize=7, loc='upper right')
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair continuous-time plots under {continuous_output_root}")

Saved 748 per-pair continuous-time plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous


## Alternation / Switch overview plots (continuous-time)

Same idea as the node-based overview: overlay every pair of a class on the shared
normalized-time axis, one figure per good unit. Saved under
`resources/outputs/{session}/direct_path_neural_continuous/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [ ]:
for path_class in ['alternation', 'switch']:
    class_dir = continuous_output_root / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, ax = plt.subplots(figsize=(8, 5))

        for pair_label in relevant_pairs:
            stats = continuous_pair_unit_stats[(pair_label, cid)]
            from_label, to_label = pair_label
            color = TARGET_COLORS.get(to_label, 'gray')
            ls = from_to_ls.get(from_label, '-')
            mean, sem = stats['mean'], stats['sem']

            ax.plot(continuous_frac_grid, mean, color=color, linewidth=1.5, linestyle=ls,
                     label=f"{from_label}->{to_label} (n={stats['n']}, {stats['mean_duration']:.1f}s)")
            ax.fill_between(continuous_frac_grid, mean - sem, mean + sem, color=color, alpha=0.1)

        ax.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(f"Unit {cid} | {path_class.capitalize()} direct paths (continuous-time)", fontsize=10)
        ax.legend(fontsize=6, loc='upper right', ncol=2)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' continuous-time overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

Saved 34 'alternation' continuous-time overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous/alternation_overview (8 pairs overlaid)
Saved 34 'switch' continuous-time overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous/switch_overview (14 pairs overlaid)


# Additional analysis: full node + edge sequence version

The per-node version (Steps 1-5) only has firing rate at node dwell-points; the
continuous-time version captures the transit gaps too but discards element identity
entirely. This version keeps *both*: the literal node-and-edge sequence the mouse
walked, firing rate computed separately for every node dwell **and** every edge
transit.

The fine-grained node+edge traversal lives in `traversal_df.csv` (same outputs folder
as `node_summary_df.csv`), but it has no `direct`/`indirect`/pair-label columns of its
own -- the path identification only happens in `improved_node_summary_df`. So rather
than re-deriving path boundaries from scratch, we reuse the direct-path instances
already extracted in Step 1 (`direct_instances` / `pairs_to_instances`) purely for
their **frame-range boundaries** (`path_frame_start`, `path_frame_end`, the first
node's `start_frame` and the last node's `end_frame`), and slice every `traversal_df`
row -- node *and* edge -- whose own frame range falls inside that span. That's the
"matching" step: it happens once per path instance via a frame-range filter, not
row-by-row.

One consequence: instance length is no longer fixed even within a pair (small
untracked single-frame gaps or brief node-edge-node wobbles add/remove elements
between traversals), so the same canonical-sequence + fractional-position resampling
used elsewhere is doing real work here.

In [73]:
traversal_csv = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/traversal_df.csv"
traversal_df = pd.read_csv(traversal_csv).sort_values('start_frame_global').reset_index(drop=True)

print(f"Traversal rows (nodes + edges): {len(traversal_df)}")
print(f"  node rows: {(traversal_df['location_type'] == 'node').sum()}")
print(f"  edge rows: {(traversal_df['location_type'] == 'edge').sum()}")

# Sanity check: traversal_df and improved_node_summary_df should describe the same
# underlying session (same overall frame span / trial count) even though the files
# may have been generated by different pipeline runs.
print(f"traversal_df max end_frame_global: {traversal_df['end_frame_global'].max()}  "
      f"vs node_summary max end_frame: {improved_node_summary_df['end_frame'].max()}")
print(f"traversal_df max trial_idx: {traversal_df['trial_idx'].max()}  "
      f"vs node_summary max trial_idx: {improved_node_summary_df['trial_idx'].max()}")

Traversal rows (nodes + edges): 10451
  node rows: 6309
  edge rows: 4142
traversal_df max end_frame_global: 145416  vs node_summary max end_frame: 145416
traversal_df max trial_idx: 736  vs node_summary max trial_idx: 736


## Build the matched full element sequence per direct-path instance

For each instance already in `direct_instances`, slice every `traversal_df` row whose
`[start_frame_global, end_frame_global]` falls within `[path_frame_start, path_frame_end]`.
Results are stored back onto the same instance dicts (new keys only -- the per-node
fields from Step 1 are untouched), so `pairs_to_instances` automatically picks them up
too.

In [74]:
def edge_label_str(raw):
    """'(\'L31\', \'L43\')' -> 'L31-L43'"""
    try:
        a, b = ast.literal_eval(raw)
        return f"{a}-{b}"
    except (ValueError, SyntaxError, TypeError):
        return str(raw)

def build_full_element_sequence(instance):
    """Every traversal_df row (node or edge) inside the instance's frame span, in
    chronological order, as parallel (type, label, start_abs, end_abs, duration_sec)
    arrays."""
    frame_start = instance['path_frame_start']
    frame_end = instance['path_frame_end']
    sub = traversal_df[
        (traversal_df['start_frame_global'] >= frame_start) &
        (traversal_df['end_frame_global'] <= frame_end)
    ]

    types, labels = [], []
    for _, r in sub.iterrows():
        if r['location_type'] == 'node':
            types.append('node')
            labels.append(r['headstage_graph_node'])
        else:
            types.append('edge')
            labels.append(edge_label_str(r['headstage_graph_edge']))

    starts_abs = sub['start_frame_global'].to_numpy() / FPS
    durations_sec = sub['duration'].to_numpy() / FPS
    ends_abs = starts_abs + durations_sec
    return types, labels, starts_abs, ends_abs, durations_sec

for inst in direct_instances:
    types, labels, starts_abs_full, ends_abs_full, durations_full = build_full_element_sequence(inst)
    inst['full_seq_types'] = types
    inst['full_seq'] = labels
    inst['full_seq_starts_abs'] = starts_abs_full
    inst['full_seq_ends_abs'] = ends_abs_full
    inst['full_seq_durations'] = durations_full
    inst['n_full_elements'] = len(labels)

print(f"Built full node+edge sequences for {len(direct_instances)} instances")
print(f"Elements per instance: min={min(i['n_full_elements'] for i in direct_instances)}, "
      f"max={max(i['n_full_elements'] for i in direct_instances)}, "
      f"mean={np.mean([i['n_full_elements'] for i in direct_instances]):.1f} "
      f"(vs. {np.mean([i['n_nodes'] for i in direct_instances]):.1f} nodes-only)")

Built full node+edge sequences for 426 instances
Elements per instance: min=13, max=85, mean=17.7 (vs. 9.4 nodes-only)


In [75]:
# Sanity check: collapsing the matched full sequence down to its node-only,
# consecutive-duplicates-merged entries should reproduce Step 1's node_seq exactly.
n_mismatch = 0
n_mismatch_trailing_only = 0
for inst in direct_instances:
    collapsed_nodes = []
    for t, l in zip(inst['full_seq_types'], inst['full_seq']):
        if t == 'node' and (not collapsed_nodes or collapsed_nodes[-1] != l):
            collapsed_nodes.append(l)
    if collapsed_nodes != inst['node_seq']:
        n_mismatch += 1
        # Most mismatches turn out to be a brief (1-frame) destination-port touch-and-back
        # wobble that node_summary_df's newer pipeline run resolved differently than
        # traversal_df's older one -- i.e. everything up to near the very end agrees, and
        # only the last node (or two) differs / is missing.
        prefix_len = 0
        for a, b in zip(collapsed_nodes, inst['node_seq']):
            if a != b:
                break
            prefix_len += 1
        shorter_len = min(len(collapsed_nodes), len(inst['node_seq']))
        if prefix_len >= shorter_len - 1:
            n_mismatch_trailing_only += 1

print(f"Instances where the frame-matched node sequence diverges from node_summary_df's: {n_mismatch} / {len(direct_instances)}")
print(f"  of which only the trailing (destination) node differs: {n_mismatch_trailing_only}")

Instances where the frame-matched node sequence diverges from node_summary_df's: 36 / 426
  of which only the trailing (destination) node differs: 34


**On the ~8% mismatch above:** `traversal_df.csv` (7 Jun) predates `node_summary_df.csv`
(15 Jun) -- different runs of the per-frame node-detection pipeline, not a bug in the
frame-range matching here. They agree on the overall route almost everywhere; the
disagreements are brief (often 1-frame) touch-and-back wobbles right at a port boundary,
where the older run's segmentation doesn't exactly match the newer one's path-boundary
call. If exact node-for-node consistency matters, regenerate `traversal_df.csv` from the
current pipeline; for the aggregate firing-rate plots below, a handful of 1-frame
boundary disagreements out of ~18 elements per path is negligible.

## Per-element firing rate + canonical full sequence

Same approach as the per-node version: `compute_full_element_rates` gives one rate
value per element (node *or* edge) for one unit; `canonical_full_sequence` picks the
most common exact (type, label) sequence for a pair, used both as the x-axis labels
and the resampling target.

In [76]:
def canonical_full_sequence(instances):
    seqs = [tuple(zip(inst['full_seq_types'], inst['full_seq'])) for inst in instances]
    most_common_seq, _ = Counter(seqs).most_common(1)[0]
    types = [t for t, _ in most_common_seq]
    labels = [l for _, l in most_common_seq]
    return types, labels

def compute_full_element_rates(instance, cluster_id):
    """Firing rate (Hz) of one unit during each node/edge element of one path instance."""
    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])
    rel_starts = instance['full_seq_starts_abs'] - instance['path_start_abs']
    rel_ends = instance['full_seq_ends_abs'] - instance['path_start_abs']

    n = instance['n_full_elements']
    rates = np.empty(n)
    for i in range(n):
        count = np.sum((rel_spikes >= rel_starts[i]) & (rel_spikes < rel_ends[i]))
        rates[i] = count / instance['full_seq_durations'][i]
    return rates

## Aggregate mean +/- SEM per (pair, good unit), full node+edge version

For **alternation** pairs, all traversals are expected to follow the same canonical
node+edge sequence (9 nodes with physical edges between the 7 internal nodes and
virtual connections at source/target ports that produce no edge element). Rather than
resampling, each instance is matched to the canonical sequence by element identity
(greedy forward scan): positions where the instance is missing an element get NaN, so
the per-position mean and SEM only average instances that actually had that element.
Non-conforming instances (sequence differs from canonical) are counted and reported.

For **switch** pairs, paths have genuinely different canonical lengths across pairs, so
resampling onto the canonical grid is still used (same as before).

In [77]:
def align_to_canonical(instance, canon_types, canon_labels, cluster_id):
    """Per-canonical-position firing rate found by greedy forward name-matching.
    Canonical positions with no matching element in the instance are returned as NaN."""
    rates = compute_full_element_rates(instance, cluster_id)
    inst_types = instance['full_seq_types']
    inst_labels = instance['full_seq']
    L = len(canon_labels)
    row = np.full(L, np.nan)
    inst_idx = 0
    for canon_pos, (ctype, clabel) in enumerate(zip(canon_types, canon_labels)):
        for j in range(inst_idx, len(inst_labels)):
            if inst_labels[j] == clabel and inst_types[j] == ctype:
                row[canon_pos] = rates[j]
                inst_idx = j + 1
                break
    return row

full_pair_unit_stats = {}
skipped_full_pairs = []

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        skipped_full_pairs.append((pair_label, len(instances)))
        continue

    canon_types, canon_labels = canonical_full_sequence(instances)
    L = len(canon_labels)
    path_class = instances[0]['path_class']

    canon_set = tuple(zip(canon_types, canon_labels))
    n_nonconforming = sum(
        1 for inst in instances
        if tuple(zip(inst['full_seq_types'], inst['full_seq'])) != canon_set
    )

    for cid in good_cluster_ids:
        if path_class == 'alternation':
            # Identity-match each instance to the canonical positions.
            # Missing positions contribute NaN; per-position stats average only
            # instances that had that element.
            rows = np.vstack([
                align_to_canonical(inst, canon_types, canon_labels, cid)
                for inst in instances
            ])
            counts = np.sum(~np.isnan(rows), axis=0)
            mean = np.nanmean(rows, axis=0)
            sem = np.nanstd(rows, axis=0, ddof=1) / np.sqrt(np.maximum(counts, 1))
        else:
            # Switch: heterogeneous canonical lengths across pairs; resample to grid.
            rows = np.vstack([
                resample_to_grid(compute_full_element_rates(inst, cid), L)
                for inst in instances
            ])
            mean = rows.mean(axis=0)
            sem = rows.std(axis=0, ddof=1) / np.sqrt(rows.shape[0])

        full_pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'n_nonconforming': n_nonconforming,
            'canon_types': canon_types,
            'canon_labels': canon_labels,
            'path_class': path_class,
        }

print(f"Computed full-sequence stats for {len(full_pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue
    stats0 = full_pair_unit_stats.get((pair_label, good_cluster_ids[0]))
    if stats0 and stats0['path_class'] == 'alternation' and stats0['n_nonconforming'] > 0:
        from_l, to_l = pair_label
        print(f"  {from_l}->{to_l}: {stats0['n_nonconforming']}/{len(instances)} instances deviate from canonical full sequence")

Computed full-sequence stats for 22 pairs x 34 good units
  Target6->Target5: 32/36 instances deviate from canonical full sequence
  Target8->Target7: 34/52 instances deviate from canonical full sequence
  Target7->Target8: 39/53 instances deviate from canonical full sequence
  Target4->Target3: 48/56 instances deviate from canonical full sequence
  Target3->Target4: 37/46 instances deviate from canonical full sequence
  Target2->Target1: 39/47 instances deviate from canonical full sequence
  Target5->Target6: 32/38 instances deviate from canonical full sequence
  Target1->Target2: 24/30 instances deviate from canonical full sequence


## Per-pair plots (full node+edge sequence)

X-axis is the canonical node+edge sequence (edge labels shown as `A-B`, italicized,
with a light gray background band, to visually separate transit elements from node
dwell elements). Saved under
`resources/outputs/{session}/direct_path_neural_with_edges/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [ ]:
output_root_edges = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_with_edges")
output_root_edges.mkdir(parents=True, exist_ok=True)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = output_root_edges / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        stats = full_pair_unit_stats[(pair_label, cid)]
        canon_types, canon_labels = stats['canon_types'], stats['canon_labels']
        L = len(canon_labels)
        mean, sem, n = stats['mean'], stats['sem'], stats['n']
        n_nc = stats['n_nonconforming']
        c = TARGET_COLORS.get(to_label, 'steelblue')

        nc_note = f", {n_nc} non-conforming" if n_nc > 0 else ""
        fig, ax = plt.subplots(figsize=(max(6, L * 0.4), 4))
        x = np.arange(L)
        for i, t in enumerate(canon_types):
            if t == 'edge':
                ax.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)
        ax.plot(x, mean, color=c, linewidth=1.8, zorder=2)
        ax.fill_between(x, mean - sem, mean + sem, color=c, alpha=0.1, zorder=1, label='SEM')
        ax.set_xticks(x)
        ax.set_xticklabels(canon_labels, rotation=90, fontsize=6)
        for tick, t in zip(ax.get_xticklabels(), canon_types):
            if t == 'edge':
                tick.set_style('italic')
                tick.set_color('dimgray')
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_xlabel('Path element (node = plain, edge = italic gray)')
        ax.set_title(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label} | n={n} traversals{nc_note}",
            fontsize=9,
        )
        ax.legend(fontsize=7, loc='upper right')
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair full-sequence plots under {output_root_edges}")

Saved 748 per-pair full-sequence plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges


## Alternation / Switch overview plots (full node+edge sequence)

Same normalized path-position overlay as the per-node overview, but built from the
full (node+edge) per-pair means -- pairs differ even more in length now that edges are
counted too. Saved under
`resources/outputs/{session}/direct_path_neural_with_edges/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [ ]:
for path_class in ['alternation', 'switch']:
    class_dir = output_root_edges / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For alternation: force a node-indexed x-axis using the most common canonical
    # full-sequence length as the expected structure (e.g. 9 nodes + 6 edges = 15
    # elements). Pairs whose canonical length differs are resampled to that expected
    # length rather than falling back to the normalized 0-1 axis.
    use_node_indexed = False
    L_expected = None
    node_x_vals = node_x_tick_positions = node_x_tick_labels = None
    alt_canon_types = None
    total_nonconforming = 0
    if path_class == 'alternation' and relevant_pairs:
        all_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                       for pl in relevant_pairs]
        L_expected = Counter(all_lengths).most_common(1)[0][0]
        mid = (L_expected - 1) // 2
        node_x_vals = np.arange(L_expected)
        node_x_tick_positions = [0, mid, L_expected - 1]
        node_x_tick_labels = ['Source\nPort', 'Mid\nNode', 'Target\nPort']
        for pl in relevant_pairs:
            if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_expected:
                alt_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
                break
        total_nonconforming = sum(
            full_pair_unit_stats[(pl, good_cluster_ids[0])]['n_nonconforming']
            for pl in relevant_pairs
        )
        use_node_indexed = True

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, ax = plt.subplots(figsize=(8, 5))

        # Gray background bands for edge-transit positions
        if use_node_indexed and alt_canon_types is not None:
            for i, t in enumerate(alt_canon_types):
                if t == 'edge':
                    ax.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

        for pair_label in relevant_pairs:
            stats = full_pair_unit_stats[(pair_label, cid)]
            from_label, to_label = pair_label
            color = TARGET_COLORS.get(to_label, 'gray')
            ls = from_to_ls.get(from_label, '-')
            L = len(stats['canon_labels'])

            if use_node_indexed:
                if L == L_expected:
                    mean_plot, sem_plot = stats['mean'], stats['sem']
                else:
                    mean_plot = resample_to_grid(stats['mean'], L_expected)
                    sem_plot = resample_to_grid(stats['sem'], L_expected)
                x_plot = node_x_vals
            else:
                mean_plot = resample_to_grid(stats['mean'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['mean']
                sem_plot = resample_to_grid(stats['sem'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['sem']
                x_plot = frac_grid

            ax.plot(x_plot, mean_plot, color=color, linewidth=1.5, linestyle=ls,
                     label=f"{from_label}->{to_label} (n={stats['n']})")
            ax.fill_between(x_plot, mean_plot - sem_plot, mean_plot + sem_plot,
                             color=color, alpha=0.1)

        if use_node_indexed:
            ax.set_xticks(node_x_tick_positions)
            ax.set_xticklabels(node_x_tick_labels)
            ax.set_xlabel('Path element position (gray = edge transit)')
            nc_note = f" | {total_nonconforming} non-conforming" if total_nonconforming > 0 else ""
        else:
            ax.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')
            nc_note = ""
        ax.set_ylabel('Firing rate (Hz)')
        ax.set_title(f"Unit {cid} | {path_class.capitalize()} direct paths (node+edge){nc_note}", fontsize=10)
        ax.legend(fontsize=6, loc='upper right', ncol=2)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' full-sequence overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

## Save full-sequence pair summary

In [80]:
full_summary_rows = []
for pair_label, instances in pairs_to_instances.items():
    from_label, to_label = pair_label
    canon_types, canon_labels = canonical_full_sequence(instances)
    full_summary_rows.append({
        'from_port': from_label,
        'to_port': to_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'n_full_elements_canonical': len(canon_labels),
        'n_nodes_canonical': len(instances[0]['node_seq']),
        'used_in_aggregation': len(instances) >= MIN_INSTANCES,
        'canonical_full_sequence': ' -> '.join(canon_labels),
    })

full_path_pair_summary_df = pd.DataFrame(full_summary_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
out_csv = output_root_edges / 'direct_path_pair_summary_with_edges.csv'
full_path_pair_summary_df.to_csv(out_csv, index=False)
print(f"Saved full-sequence pair summary to {out_csv}")
display(full_path_pair_summary_df)

Saved full-sequence pair summary to /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges/direct_path_pair_summary_with_edges.csv


,from_port,to_port,path_class,n_instances,n_full_elements_canonical,n_nodes_canonical,used_in_aggregation,canonical_full_sequence
0,Target4,Target3,alternation,56,16,9,True,L510 -> L510 -> L45 -> L32-L45 -> L32 -> L21-L...
1,Target7,Target8,alternation,53,13,9,True,R59 -> R44 -> R32-R44 -> R32 -> R21-R32 -> R21...
2,Target8,Target7,alternation,52,15,9,True,R55 -> R42 -> R31-R42 -> R31 -> R20-R31 -> R20...
3,Target2,Target1,alternation,47,15,9,True,R526 -> R413 -> R36-R413 -> R36 -> R23-R36 -> ...
4,Target3,Target4,alternation,46,15,9,True,L56 -> L43 -> L31-L43 -> L31 -> L20-L31 -> L20...
5,Target5,Target6,alternation,38,16,9,True,L521 -> L521 -> L410 -> L35-L410 -> L35 -> L22...
6,Target6,Target5,alternation,36,15,9,True,L525 -> L412 -> L36-L412 -> L36 -> L23-L36 -> ...
7,Target1,Target2,alternation,30,16,9,True,R522 -> R411 -> R35-R411 -> R35 -> R22-R35 -> ...
8,Target2,Target8,switch,12,19,11,True,R526 -> R413 -> R36-R413 -> R36 -> R23-R36 -> ...
9,Target5,Target4,switch,7,22,11,True,L521 -> L521 -> L521 -> L410 -> L35-L410 -> L3...


# Combined 3-panel overview: continuous / node+edge / node-only

One figure per good unit per path class with three stacked panels:

- **Top**: continuous-time binned firing rate (normalized path time)
- **Middle**: node + edge sequence firing rate (element-position x-axis for alternation, gray bands = edge transits)
- **Bottom**: node-only firing rate (node-position x-axis for alternation)

For alternation the middle and bottom x-axes are structural (Source Port → Mid Node → Target Port). For switch they stay normalized 0–1. The figure title notes the total number of non-conforming alternation instances across all pairs.

In [ ]:
combined_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_combined")
combined_output_root.mkdir(parents=True, exist_ok=True)

for path_class in ['alternation', 'switch']:
    combined_dir = combined_output_root / f"{path_class}_overview"
    combined_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]
    if not relevant_pairs:
        continue

    # For switch: linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    # Node-indexed x-axis setup for alternation (middle and bottom panels)
    use_node_indexed = False
    L_nodes_exp = L_full_exp = mid_node = mid_full = None
    alt_full_canon_types = None
    total_nc = 0
    if path_class == 'alternation':
        node_lengths = [len(pair_unit_stats[(pl, good_cluster_ids[0])]['canonical_seq'])
                        for pl in relevant_pairs]
        L_nodes_exp = Counter(node_lengths).most_common(1)[0][0]
        mid_node = (L_nodes_exp - 1) // 2

        full_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                        for pl in relevant_pairs]
        L_full_exp = Counter(full_lengths).most_common(1)[0][0]
        mid_full = (L_full_exp - 1) // 2

        for pl in relevant_pairs:
            if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_full_exp:
                alt_full_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
                break

        total_nc = sum(
            full_pair_unit_stats[(pl, good_cluster_ids[0])]['n_nonconforming']
            for pl in relevant_pairs
        )
        use_node_indexed = True

    n_saved = 0
    for cid in good_cluster_ids:
        fig, axes = plt.subplots(3, 1, figsize=(9, 11), sharey=True)
        ax_cont, ax_full, ax_node = axes

        # Gray edge-transit bands on the node+edge panel (alternation only)
        if use_node_indexed and alt_full_canon_types is not None:
            for i, t in enumerate(alt_full_canon_types):
                if t == 'edge':
                    ax_full.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

        legend_handles, legend_labels = [], []
        for pair_label in relevant_pairs:
            from_label, to_label = pair_label
            color = TARGET_COLORS.get(to_label, 'gray')
            ls = from_to_ls.get(from_label, '-')

            # ── Top: continuous ──────────────────────────────────────────────────
            sc = continuous_pair_unit_stats[(pair_label, cid)]
            line, = ax_cont.plot(continuous_frac_grid, sc['mean'], color=color, linewidth=1.5, linestyle=ls)
            ax_cont.fill_between(continuous_frac_grid, sc['mean'] - sc['sem'],
                                  sc['mean'] + sc['sem'], color=color, alpha=0.1)
            legend_handles.append(line)
            legend_labels.append(f"{from_label}->{to_label} (n={sc['n']})")

            # ── Middle: node+edge ─────────────────────────────────────────────────
            sf = full_pair_unit_stats[(pair_label, cid)]
            L_f = len(sf['canon_labels'])
            if use_node_indexed:
                mf = sf['mean'] if L_f == L_full_exp else resample_to_grid(sf['mean'], L_full_exp)
                semf = sf['sem'] if L_f == L_full_exp else resample_to_grid(sf['sem'], L_full_exp)
                xf = np.arange(L_full_exp)
            else:
                mf = resample_to_grid(sf['mean'], OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['mean']
                semf = resample_to_grid(sf['sem'], OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['sem']
                xf = frac_grid
            ax_full.plot(xf, mf, color=color, linewidth=1.5, linestyle=ls)
            ax_full.fill_between(xf, mf - semf, mf + semf, color=color, alpha=0.1)

            # ── Bottom: node-only ─────────────────────────────────────────────────
            sn = pair_unit_stats[(pair_label, cid)]
            L_n = len(sn['canonical_seq'])
            if use_node_indexed:
                mn = sn['mean'] if L_n == L_nodes_exp else resample_to_grid(sn['mean'], L_nodes_exp)
                semn = sn['sem'] if L_n == L_nodes_exp else resample_to_grid(sn['sem'], L_nodes_exp)
                xn = np.arange(L_nodes_exp)
            else:
                mn = resample_to_grid(sn['mean'], OVERVIEW_GRID) if L_n != OVERVIEW_GRID else sn['mean']
                semn = resample_to_grid(sn['sem'], OVERVIEW_GRID) if L_n != OVERVIEW_GRID else sn['sem']
                xn = frac_grid
            ax_node.plot(xn, mn, color=color, linewidth=1.5, linestyle=ls)
            ax_node.fill_between(xn, mn - semn, mn + semn, color=color, alpha=0.1)

        # ── Format axes ───────────────────────────────────────────────────────────
        ax_cont.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
        ax_cont.set_ylabel('Firing rate (Hz)')
        ax_cont.set_title('Continuous time', fontsize=9)

        ax_full.set_ylabel('Firing rate (Hz)')
        ax_full.set_title('Node + edge', fontsize=9)
        if use_node_indexed:
            ax_full.set_xticks([0, mid_full, L_full_exp - 1])
            ax_full.set_xticklabels(['Source\nPort', 'Mid\nNode', 'Target\nPort'])
            ax_full.set_xlabel('Path element position (gray = edge transit)')
        else:
            ax_full.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')

        ax_node.set_ylabel('Firing rate (Hz)')
        ax_node.set_title('Node only', fontsize=9)
        if use_node_indexed:
            ax_node.set_xticks([0, mid_node, L_nodes_exp - 1])
            ax_node.set_xticklabels(['Source\nPort', 'Mid\nNode', 'Target\nPort'])
            ax_node.set_xlabel('Path node position')
        else:
            ax_node.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')

        nc_note = f" | {total_nc} non-conforming" if total_nc > 0 else ""
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths{nc_note}",
                     fontsize=11, fontweight='bold')

        # Legend below all subplots, outside the plot area
        n_cols = min(len(legend_labels), 4)
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.09, top=0.94)
        fig.legend(legend_handles, legend_labels,
                   loc='lower center', bbox_to_anchor=(0.5, 0),
                   fontsize=6, ncol=n_cols, frameon=True)

        fig.savefig(combined_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} combined '{path_class}' overview plots under {combined_dir}")

Saved 34 combined 'alternation' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_combined/alternation_overview
Saved 34 combined 'switch' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_combined/switch_overview
